# 🧪 Confluence 문서 파싱 & 청킹 실험 노트북

이 노트북은 Confluence REST API의 HTML 데이터, 표(Table), 링크, 그리고 파싱된 텍스트의 청킹(Chunking) 결과를 눈으로 빠르게 실험하고 검증하기 위한 용도입니다.

In [ ]:
import sys
import os

# ai-server 경로 추가
sys.path.append('../ai-server')

from app.parser.confluence_parser import parse_confluence_html, split_text_into_chunks
from app.core.confluence_client import fetch_confluence_pages

## 1. Confluence 샘플 HTML 파싱 테스트 (표 & 링크 보존 검증)

Confluence REST API가 돌려주는 실제 HTML 샘플을 넣어서, 표(`<table>`)가 마크다운 표(`| 제목 | 내용 |`)로 잘 변환되는지 확인합니다.

In [ ]:
sample_html = '''
<h1>A 시스템 통합 운영 가이드</h1>
<p>본 문서는 A 시스템의 메인 운영 지침서입니다. 상세한 사항은 <a href="https://lloydk.atlassian.net/wiki/pages/102">시스템 상세 페이지</a>를 참고하세요.</p>

<h3>시스템 담당자 목록</h3>
<table>
  <tr><th>구분</th><th>담당자</th><th>소속 팀</th><th>비고</th></tr>
  <tr><td>메인 서버</td><td>김철수 수석</td><td>DE Platform 팀</td><td>주 담당자</td></tr>
  <tr><td>DB 관리</td><td>이영희 책임</td><td>Data Eng 팀</td><td>부 담당자</td></tr>
</table>
'''

# 파서 실행
parsed_result = parse_confluence_html(sample_html, metadata={"space": "LLOYDK"})

print("=== [1] 파싱된 텍스트 (표가 마크다운으로 변환됨) ===")
print(parsed_result["cleaned_text"])
print("\n=== [2] 추출된 링크 목록 ===")
print(parsed_result["links"])

## 2. 청킹(Chunking) 분할 및 크기 테스트

파싱된 텍스트를 RAG 검색용 청크(Chunk)로 나눴을 때 오버랩(Overlap)과 텍스트 분할이 적절한지 검증합니다.

In [ ]:
chunks = split_text_into_chunks(
    doc_id="doc-101",
    title="A 시스템 통합 운영 가이드",
    text=parsed_result["cleaned_text"],
    chunk_size=200,
    chunk_overlap=40
)

print(f"총 분할된 청크 개수: {len(chunks)}개\n")
for chunk in chunks:
    print(f"--- [청크 ID: {chunk['chunk_id']}] ---")
    print(chunk['text'])
    print()

## 3. 실제 Confluence API 데이터 수집 및 파싱 테스트 (.env 설정 필요)

실제 Confluence 접속 정보가 `.env`에 설정되어 있다면 실제 문서 1건을 끌어와서 파싱해봅니다.

In [ ]:
try:
    real_pages = fetch_confluence_pages()
    if real_pages:
        page = real_pages[0]
        print(f"[수집 성공] 제목: {page['title']}")
        real_parsed = parse_confluence_html(page['html_body'], metadata={"id": page['id']})
        print("\n--- 파싱된 실제 텍스트 샘플 (상위 300자) ---")
        print(real_parsed['cleaned_text'][:300])
    else:
        print("수집된 문서가 없거나 인증 정보(.env)를 확인해주세요.")
except Exception as e:
    print(f"실제 연동 테스트 스킵 (사유: {e})")